# Lab 03: Machine Learning on Spark
> Classification with Logistic Regression - Low-level Operations

## Preprocess

### Web Crawl

Download the dataset straight from Kaggle's server using `cURL` and unzip the downloaded file using Python's native `unzip` library.

In [ ]:
!curl -L -o ./creditcardfraud.zip\
  https://www.kaggle.com/api/v1/datasets/download/mlg-ulb/creditcardfraud

!python -m zipfile -e 'creditcardfraud.zip' './'
!rm 'creditcardfraud.zip'

### Setup Spark Environment

Import `findspark` to find paths (e.g. `SPARK_HOME`) and setup environment for data processing with `pyspark`.

In [1]:
import findspark
findspark.init()

Create a `SparkSession` to work on RDDs.

In [2]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
                    .master('local[*]') \
                    .appName("ClassificationRDD") \
                    .config("spark.some.config.option", "some-value")\
                    .getOrCreate()

sc = spark.sparkContext

25/05/10 16:02:35 WARN Utils: Your hostname, DESKTOP-OECCCK2 resolves to a loopback address: 127.0.1.1; using 172.28.205.125 instead (on interface eth0)
25/05/10 16:02:35 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/05/10 16:02:37 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Supress non-lethal I/O warnings.

In [3]:
log4j = sc._jvm.org.apache.log4j

log4j.Logger.getLogger("org.apache.hadoop.hdfs.client.impl.BlockReaderFactory") \
     .setLevel(log4j.Level.ERROR)

### Import & Format Dataset From `.csv` File.

Import the dataset from local directory into a RDD. 

Format the RDD's values to its corressponding data types:
- Features: From string to float.
- Label: From string to int (0/1), remove the excecss formats.

In [4]:
import os

lines = sc.textFile(f"file:///{os.getcwd()}/creditcard.csv") 

header = lines.first() 
data = lines.filter(lambda line: line != header)

parsedData = data.map(lambda line: 
    [float(x) for x in line.split(",")[:-1]] + \
    [int(line.split(",")[-1].strip('"'))] 
)

rdd_data = parsedData.map(lambda cols: (cols[:-1], cols[-1]))

Output the RDD for examination

In [5]:
print('Headers:', header)

print('Sample data records:')
for record in rdd_data.take(3):
    print(record)

print(f"Total data records processed: {rdd_data.count()}")

Headers: "Time","V1","V2","V3","V4","V5","V6","V7","V8","V9","V10","V11","V12","V13","V14","V15","V16","V17","V18","V19","V20","V21","V22","V23","V24","V25","V26","V27","V28","Amount","Class"
Sample data records:


([0.0, -1.3598071336738, -0.0727811733098497, 2.53634673796914, 1.37815522427443, -0.338320769942518, 0.462387777762292, 0.239598554061257, 0.0986979012610507, 0.363786969611213, 0.0907941719789316, -0.551599533260813, -0.617800855762348, -0.991389847235408, -0.311169353699879, 1.46817697209427, -0.470400525259478, 0.207971241929242, 0.0257905801985591, 0.403992960255733, 0.251412098239705, -0.018306777944153, 0.277837575558899, -0.110473910188767, 0.0669280749146731, 0.128539358273528, -0.189114843888824, 0.133558376740387, -0.0210530534538215, 149.62], 0)
([0.0, 1.19185711131486, 0.26615071205963, 0.16648011335321, 0.448154078460911, 0.0600176492822243, -0.0823608088155687, -0.0788029833323113, 0.0851016549148104, -0.255425128109186, -0.166974414004614, 1.61272666105479, 1.06523531137287, 0.48909501589608, -0.143772296441519, 0.635558093258208, 0.463917041022171, -0.114804663102346, -0.183361270123994, -0.145783041325259, -0.0690831352230203, -0.225775248033138, -0.638671952771851, 0

[Stage 2:===========>                                               (1 + 4) / 5]

Total data records processed: 284807


Normalize the `Amount` column

In [6]:
amounts = rdd_data.map(lambda fl: fl[0][-1])
min_amt = amounts.min()
max_amt = amounts.max()
range_amt = max_amt - min_amt

rdd_normalized = rdd_data.map(lambda fl: (
    fl[0][:-1]                                          
    + [(fl[0][-1] - min_amt) / range_amt],
    fl[1] 
))

for record in rdd_normalized.take(3):
    print(record)

([0.0, -1.3598071336738, -0.0727811733098497, 2.53634673796914, 1.37815522427443, -0.338320769942518, 0.462387777762292, 0.239598554061257, 0.0986979012610507, 0.363786969611213, 0.0907941719789316, -0.551599533260813, -0.617800855762348, -0.991389847235408, -0.311169353699879, 1.46817697209427, -0.470400525259478, 0.207971241929242, 0.0257905801985591, 0.403992960255733, 0.251412098239705, -0.018306777944153, 0.277837575558899, -0.110473910188767, 0.0669280749146731, 0.128539358273528, -0.189114843888824, 0.133558376740387, -0.0210530534538215, 0.0058237930868049554], 0)
([0.0, 1.19185711131486, 0.26615071205963, 0.16648011335321, 0.448154078460911, 0.0600176492822243, -0.0823608088155687, -0.0788029833323113, 0.0851016549148104, -0.255425128109186, -0.166974414004614, 1.61272666105479, 1.06523531137287, 0.48909501589608, -0.143772296441519, 0.635558093258208, 0.463917041022171, -0.114804663102346, -0.183361270123994, -0.145783041325259, -0.0690831352230203, -0.225775248033138, -0.638

## Modeling

### Logistic Regression Model Using SGD

The `ClassifierSGD` class implements methods and features of a Logistic Regression model, learning using Stochastic gradient descent (SGD).

In [12]:
class SparkSGDClassifier:
    def __init__(self, learning_rate=0.01, num_iterations=20, seed=42):
        self.learning_rate = learning_rate
        self.num_iterations = num_iterations
        self.seed = seed
        self.weights = None
        self.bias = 0.0

    def fit(self, data_rdd):
        """
        Train logistic regression via batch gradient descent on RDD.
        Uses broadcast variables and mapPartitions to parallelize gradient computation.
        """
        # Initialize weights
        first = data_rdd.first()
        num_features = len(first[0])
        
        random.seed(self.seed)
        
        self.weights = [random.uniform(-0.01, 0.01) for _ in range(num_features)]
        self.bias = 0.0

        # Cache and count samples
        data_rdd = data_rdd.cache()
        n_samples = data_rdd.count()

        for iteration in range(self.num_iterations):
            # Broadcast current parameters
            sc = data_rdd.context
            bc_w = sc.broadcast(self.weights)
            bc_b = sc.broadcast(self.bias)

            def partition_grad(partition):
                w = bc_w.value
                b = bc_b.value
                grad_w = [0.0] * len(w)
                grad_b = 0.0
                
                count = 0
                
                for features, label in partition:
                    count += 1
                
                    z = sum(f * w_i for f, w_i in zip(features, w)) + b

                    if z < -100:
                        pred = 0.0
                    elif z > 100:
                        pred = 1.0
                    else:
                        pred = 1.0 / (1.0 + math.exp(-z))
                        
                    error = pred - label

                    for i, f in enumerate(features):
                        grad_w[i] += error * f
                    grad_b += error
                
                if count:
                    yield grad_w, grad_b, count

            # Sum gradients across partitions
            grad_w_sum, grad_b_sum, total_count = (
                data_rdd
                .mapPartitions(partition_grad)
                .reduce(lambda a, b: (
                    [x + y for x, y in zip(a[0], b[0])],
                    a[1] + b[1],
                    a[2] + b[2]
                ))
            )

            # Compute average gradient
            avg_grad_w = [gw / total_count for gw in grad_w_sum]
            avg_grad_b = grad_b_sum / total_count

            # Parameter update
            self.weights = [w - self.learning_rate * gw for w, gw in zip(self.weights, avg_grad_w)]
            self.bias -= self.learning_rate * avg_grad_b

            # Unpersist broadcasts
            bc_w.unpersist()
            bc_b.unpersist()

        return self

    def predict(self, features_rdd):
        """
        Predict binary labels for an RDD of feature vectors.
        """
        sc = features_rdd.context
        bc_w = sc.broadcast(self.weights)
        bc_b = sc.broadcast(self.bias)

        def _predict_point(features):
            w = bc_w.value
            b = bc_b.value
            z = sum(f * w_i for f, w_i in zip(features, w)) + b
            # stable sigmoid
            if z < -100:
                prob = 0.0
            elif z > 100:
                prob = 1.0
            else:
                prob = 1.0 / (1.0 + math.exp(-z))
            return 1 if prob >= 0.01 else 0

        preds = features_rdd.map(_predict_point)
        bc_w.unpersist()
        bc_b.unpersist()
        return preds

    def get_weights(self):
        return self.weights


### Metrics

The `MetricEvaluator` class provides methods to get metrics related to the model's performance by comparing the model's prediction with the test set's actual label.

In [9]:
class MetricEvaluator:
    def __init__(self, prediction_rdd, label_rdd):
        """
        Efficiently compute confusion matrix components in one pass over zipped RDDs.
        """
        # Zip labels and predictions, cache for reuse
        paired = label_rdd.zip(prediction_rdd).cache()

        # Aggregate counts for TP, FP, FN, TN
        tp, fp, fn, tn = paired.map(
            lambda lp: (
                1 if lp[0] == 1 and lp[1] == 1 else 0,
                1 if lp[0] == 0 and lp[1] == 1 else 0,
                1 if lp[0] == 1 and lp[1] == 0 else 0,
                1 if lp[0] == 0 and lp[1] == 0 else 0
            )
        ).reduce(lambda a, b: (a[0] + b[0], a[1] + b[1], a[2] + b[2], a[3] + b[3]))

        self._tp = tp
        self._fp = fp
        self._fn = fn
        self._tn = tn
        self.n_samples = tp + fp + fn + tn

    def accuracy(self):
        return (self._tp + self._tn) / self.n_samples if self.n_samples > 0 else 0.0

    def precision(self):
        denom = self._tp + self._fp
        return self._tp / denom if denom > 0 else 0.0

    def recall(self):
        denom = self._tp + self._fn
        return self._tp / denom if denom > 0 else 0.0

    def specificity(self):
        denom = self._tn + self._fp
        return self._tn / denom if denom > 0 else 0.0

    def f1_score(self):
        p = self.precision()
        r = self.recall()
        return 2 * p * r / (p + r) if (p + r) > 0 else 0.0

    def print_metrics(self):
        print("Metrics:")
        print(f"- Accuracy:    {self.accuracy():.4f}")
        print(f"- Precision:   {self.precision():.4f}")
        print(f"- Recall:      {self.recall():.4f}")
        print(f"- F1 Score:    {self.f1_score():.4f}")
        print(f"- Specificity: {self.specificity():.4f}")

### Train the model

In [ ]:
train_rdd, test_rdd = rdd_normalized.randomSplit([0.8, 0.2])

learning_rates = [1, 0.01, 0.001]
n_iters = [20,50,100]

for lr, it in zip(learning_rates, n_iters):
        print(f"Logistic Regression with SGD: LR={lr}, Iter={it}")
        
        sgd_clf = SparkSGDClassifier(
            learning_rate=lr,
            num_iterations=it,
        )
            
        sgd_clf.fit(train_rdd)

        print("Updated Weights:", sgd_clf.get_weights())
    
        features_test_rdd = test_rdd.map(lambda x: x[0])
        true_labels_rdd = test_rdd.map(lambda x: x[1])
        
        predictions_rdd = sgd_clf.predict(features_test_rdd)
    
        MetricEvaluator(
            predictions_rdd,
            true_labels_rdd
        ).print_metrics()

Logistic Regression with SGD: LR=1, Iter=20


Updated Weights: [-92084.29791350129, -0.17364293110394938, 0.11791812992280781, -0.2451326305633822, 0.16022018762490603, -0.1051766118553868, -0.03730270771138395, -0.19741227955224852, 0.0141049652282224, -0.09822730407285049, -0.20142514038904372, 0.12928813908074233, -0.2243306029575499, -0.009890649116517517, -0.23365453846702705, -0.0033606297335565674, -0.14602990422705053, -0.22536122789805607, -0.07056766920711811, 0.013031032946719727, 0.019751027464234152, 0.026037143641574667, -0.002827486561561866, -0.008064983100864296, 0.005626997469302112, -0.0013576507390465728, -0.006108009380285013, -0.0023135488661031644, 0.008887875055900748, -0.001183651522236402]


25/05/10 16:06:35 WARN BlockManager: Task 341 already completed, not releasing lock for rdd_55_0


Metrics:
- Accuracy:    0.9981
- Precision:   0.0000
- Recall:      0.0000
- F1 Score:    0.0000
- Specificity: 1.0000
Logistic Regression with SGD: LR=0.01, Iter=50


Updated Weights: [-880.4945475582394, -0.01357114463226769, -0.0014383462656181348, -0.011495384454619537, 0.008630826005875069, 0.0008399218497412546, 0.006697299337571677, -0.0130278224738783, -0.0012015210065511438, -0.011628207938151345, -0.010526273074466398, 0.0033355526918118165, -0.014837008446077469, -0.006105789494289527, -0.0029217481848941725, 0.0007884027619450549, -0.009114336394995008, -0.0038737729368573757, 0.004266237123148166, -0.009314224781181667, 0.006457329717929102, 0.004538212544962126, -0.0031786716921673986, -0.006923119081215443, 0.009056480366930832, -0.0032255497588509307, -0.008098679148339265, -0.007928981310855145, 0.0069971494119848545, 0.0020444589300448535]


25/05/10 16:07:40 WARN BlockManager: Task 602 already completed, not releasing lock for rdd_55_0


Metrics:
- Accuracy:    0.9981
- Precision:   0.0000
- Recall:      0.0000
- F1 Score:    0.0000
- Specificity: 1.0000
Logistic Regression with SGD: LR=0.001, Iter=100


[Stage 159:===========>                                             (1 + 4) / 5]

In [ ]:
spark.stop()